## 0. 개발 환경 구성

1. Python 가상 환경 생성  
   - 이름 : `python_db`
   - `python -m venv python_db`

2. VSCode에 가상 환경의 Python 등록 -> `Python: Select Interpreter`

3. 가상 환경 활성화
   - 윈도우 git bash → `source python_db/Scripts/activate`
   - 윈도우 CMD → `call python_db/Scripts/activate`
   - 맥 터미널 → `source python_db/bin/activate`

4. 가상 환경에 `requirements.txt` 파일 내 라이브러리들 설치
   - `pip install -r requirements.txt`

## 1. Python을 통한 DB Connection

In [ ]:
import sqlite3, pandas as pd, sqlalchemy, pymysql

In [ ]:
## sqlite3 <--> sqlite DB
## pymysql <--> mysql DB
## psycopg2 <--> postgresql DB
## oracle_db <--> oracle DB
## pymssql <--> mssql DB
## sqlalchemy <--> 다양한 RDBMS를 공통적인 코드로 connection 연결이 가능

### 1-1 SQLite 라이브러리(sqlite3)를 활용한 Connection

- SQLite는 서버 없이 로컬 파일 하나로 동작하는 내장형 DB입니다.
- 파일이 없으면 `connect()` 시점에 새로 생성됩니다.

In [ ]:
import sqlite3

sqlite = sqlite3.connect("pokemon.db")
sqlite

#### cursor 란?
- 쿼리문에 의해서 반환되는 결과값들을 저장하는 메모리공간

In [ ]:
cursor = sqlite.cursor()

In [ ]:
cursor.execute("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name")

In [ ]:
cursor.fetchall()

In [ ]:
# CURSOR에 저장된 결과를 fetch로 가져오면 다시 못가져온다. -> 결과 fetch는 한 번만 가능하다!

cursor.fetchone()

# cursor.fetchmany(3)

#### DDL & DML 쿼리 생성

In [ ]:
## 테이블 생성 쿼리

create_query = """
    CREATE TABLE IF NOT EXISTS lecture (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name VARCHAR(20),
        year INT,
        gender VARCHAR(10),
        count INT
        );
"""

cursor.execute(create_query)
sqlite.commit()

In [ ]:
cursor.execute("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name")
cursor.fetchall()

In [ ]:
## INSERT
insert_query = """
    INSERT INTO lecture (id, name, year, gender, count)
    VALUES (1, 'codeit', 2025, 'M', 100);
"""

cursor.execute(insert_query)

In [ ]:
## Commit -> 수정 내용 확정! 수정 내용을 물리적으로 반영!
sqlite.commit()

In [ ]:
## UPDATE
update_query = """
    UPDATE lecture SET count = 200 WHERE id = 1
"""

cursor.execute(update_query)
sqlite.commit()

In [ ]:
## DELETE
delete_query = """
    DELETE FROM lecture WHERE id = 1
"""

cursor.execute(delete_query)
sqlite.commit()

In [ ]:
cursor.close()

sqlite.close()

# 데이터베이스 connection 생성 -> cursor 생성 -> SQL 쿼리 실행 -> commit -> cursor close

#### Connection Pool

<img src="https://velog.velcdn.com/images/newnew_daddy/post/f0569aa7-1aad-466e-a24f-5e3b5f248a72/image.png" width="70%">

- 일정량의 Connection 객체를 미리 만들어서 pool에 저장
- 클라이언트 요청이 오면 Connection 객체를 빌려주고 해당 객체의 임무가 완료되면 다시 Connection 객체를 반납 받아 pool에 저장
- 큰 커넥션 풀은 메모리 소모가 큰 대신 대기 시간이 적어지고, 작은 커넥션 풀은 메모리 소모가 작은 대신 대기 시간이 길어진다. ([적정 Connection 수 공식](https://github.com/brettwooldridge/HikariCP/wiki/About-Pool-Sizing#the-formula))
- 자원을 사용하면 반드시 반납을 해줘야하는데 이를 위해 DB 연결시 파이썬 `with문`을 사용

### 1-2 MySQL 라이브러리(pymysql)를 활용한 Connection

- 맥 터미널 / 윈도우 git bash

```bash
docker run -d \
  --name mysql_db \
  -p 3300:3306 \
  -e MYSQL_ROOT_PASSWORD=123456 \
  -e MYSQL_USER=practice \
  -e MYSQL_PASSWORD=123456 \
  -e MYSQL_DATABASE=docker_mysql \
  mysql
```

- 윈도우 CMD / PowerShell (한 줄)

```bash
docker run -d --name mysql_db -p 3300:3306 -e MYSQL_ROOT_PASSWORD=123456 -e MYSQL_USER=practice -e MYSQL_PASSWORD=123456 -e MYSQL_DATABASE=docker_mysql mysql
```

| 옵션 | 의미 |
| --- | --- |
| `-d` | 백그라운드 실행 |
| `--name mysql_db` | 컨테이너 이름 |
| `-p 3300:3306` | 내 PC 3300 포트 → 컨테이너 MySQL 3306 포트 |
| `-e ...` | root 비밀번호, 접속 계정, 기본 DB 생성 |
| `mysql` | 사용할 이미지 |

- 실행 확인: `docker ps` (STATUS가 `Up`이면 정상, 최초 실행 후 10~20초 뒤 접속 가능)

In [1]:
import pymysql

user = 'practice'
password = '123456'
host = 'localhost'
port = 3300
database = 'docker_mysql'

mysql = pymysql.connect(
    user=user,
    password=password,
    host=host,
    port=port,
    database=database,
    charset='utf8'
    )

mysql

In [2]:
cursor = mysql.cursor()

In [3]:
cursor.execute("SHOW DATABASES")

3

In [4]:
cursor.fetchall()

(('docker_mysql',), ('information_schema',), ('performance_schema',))

#### DDL & DML 쿼리 생성

In [5]:
## 테이블 생성 쿼리 

create_query = """
    CREATE TABLE IF NOT EXISTS lecture (
        id INT AUTO_INCREMENT PRIMARY KEY, 
        name VARCHAR(20),
        year INT,
        gender VARCHAR(10),
        count INT
        );
"""

cursor.execute(create_query)
mysql.commit()

In [6]:
cursor.execute("SHOW TABLES FROM docker_mysql")
cursor.fetchall()

(('lecture',),)

In [7]:
## INSERT
insert_query = """
    INSERT INTO lecture
    VALUES (1, 'codeit', 2025, 'M', 100);
"""

cursor.execute(insert_query)

1

In [8]:
## Commit -> 수정 내용 확정! 수정 내용을 물리적으로 반영!
mysql.commit()

In [9]:
## UPDATE
update_query = """
    UPDATE lecture SET count = 200 WHERE id = 1
"""

cursor.execute(update_query)
mysql.commit()

In [10]:
## DELETE
delete_query = """
    DELETE FROM lecture WHERE id = 1
"""

cursor.execute(delete_query)
mysql.commit()

In [11]:
cursor.close()

mysql.close()

# 데이터베이스 connection 생성 -> cursor 생성 -> SQL 쿼리 실행 -> commit -> cursor close

### 1-3. with문 활용

#### Bad Case

<img src="https://velog.velcdn.com/images/newnew_daddy/post/df312bda-0b22-4476-8a03-505f1d3cf5b4/image.png" width="50%">

#### Good Case

<img src="https://velog.velcdn.com/images/newnew_daddy/post/707d46b0-8a0b-4862-ad95-285dc04ddc29/image.png" width="50%">

- Python에서 파일 또는 리소스 관리를 더 효과적으로 처리하기 위한 블록 구조
- 리소스를 열고 사용한 후 자동으로 닫아주기 때문에 닫아주는 코드를 작성할 필요가 없습니다. 
- 주로 파일 입출력, 데이터베이스 연결, 네트워크 연결 등 리소스 관리에 사용됩니다.

In [ ]:
mysql 
cursor = mysql.cursor()

cursor.execute('SHOW DATABASES')
cursor.fetchall()
cursor.close()
# mysql.close()

In [ ]:
mysql 
cursor = mysql.cursor()


with mysql:
    with cursor:
        cursor.execute('SHOW DATABASES')
        print(cursor.fetchall())
        # cursor 자원 반납 프로세스 진행!
        
    # 데이터베이스 연결 객체 반납

In [ ]:
with mysql as conn:
    with cursor as cur:
        cur.execute('SHOW DATABASES')
        print(cur.fetchall())
        # cursor 자원 반납 프로세스 진행!
        
    # 데이터베이스 연결 객체 반납

### 1-4. sqlalchemy 라이브러리를 활용한 Connection

연결 문자열 형식: `{engine_name}://{user}:{password}@{host}:{port}/{database}`

`engine_name`은 **dialect(+driver)** 로 구성합니다. DB 종류에 따라 아래처럼 지정하면 됩니다.

| DBMS | engine_name 예시 |
|---|---|
| SQLite | `sqlite` (파일 경로는 `sqlite:///파일경로.db`) |
| MySQL | `mysql+pymysql` |
| MariaDB | `mariadb+pymysql` |
| PostgreSQL | `postgresql+psycopg2` |
| Oracle | `oracle+oracledb` |
| SQL Server | `mssql+pyodbc` |

- dialect만 적어도 기본 드라이버를 쓰지만, MySQL은 `mysql+pymysql`처럼 **드라이버까지 명시**하는 편이 명확합니다.
- 드라이버 패키지는 별도로 설치해야 합니다. (예: MySQL → `pymysql`, PostgreSQL → `psycopg2`)
- SQLite는 표준 라이브러리 `sqlite3`를 쓰므로 별도 드라이버 설치가 필요 없습니다.

In [ ]:
from sqlalchemy import create_engine, text

## SQLITE connection (로컬 파일 DB)
engine_name = "sqlite"
database = "pokemon.db"

sqlite_conn = create_engine(f"{engine_name}:///{database}")

sqlite_conn

In [ ]:
cursor = sqlite_conn.connect()

In [ ]:
res = cursor.execute(text("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name"))

res.fetchall()

In [ ]:
## MYSQL connection
engine_name = "mysql+pymysql"
user = 'practice'
password = '123456'
host = 'localhost'
port = 3300
database = 'docker_mysql'

mysql_conn = create_engine(f"{engine_name}://{user}:{password}@{host}:{port}/{database}")

mysql_conn

In [ ]:
cursor = mysql_conn.connect()

In [ ]:
res = cursor.execute(text("SHOW DATABASES"))

res.fetchall()

### 1-5. pandas을 활용한 조회 및 저장
- sqlalchemy connection만 활용 가능!

#### 1) SQLite 연결 객체 생성

In [4]:
from sqlalchemy import create_engine

## SQLITE connection
engine_name = "sqlite"
database = "pokemon.db"

sqlite_conn = create_engine(f"{engine_name}:///{database}")

sqlite_conn

Engine(sqlite:///pokemon.db)

#### 2) 테이블 저장
- [to_sql()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_sql.html) 메소드
- dataframe.to_sql( 테이블 이름, sqlalchemy_connector, 옵션 )

In [2]:
import pandas as pd

pokemon = pd.read_csv('pokemon.csv')

pokemon.head()

,id,kor_name,eng_name,type1,type2,total,hp,attack,defense,special_attack,special_defense,speed,generation,is_legendary
0,1,이상해씨,Bulbasaur,Grass,Poison,318,45,49,49,65,65,45,1,False
1,2,이상해풀,Ivysaur,Grass,Poison,405,60,62,63,80,80,60,1,False
2,3,이상해꽃,Venusaur,Grass,Poison,525,80,82,83,100,100,80,1,False
3,4,파이리,Charmander,Fire,NaN,309,39,52,43,60,50,65,1,False
4,5,리자드,Charmeleon,Fire,NaN,405,58,64,58,80,65,80,1,False


In [14]:
pokemon.to_sql(
    name='pokemon',  # db에 저장될 테이블 이름
    con=sqlite_conn,   # sqlalchemy 연결 객체
    if_exists='replace',
    index=False
    )

251

In [15]:
trainer = pd.read_csv('trainer.csv')

trainer.to_sql(
    name='trainer',
    con=sqlite_conn,
    if_exists='replace',
    index=False
)

90

In [16]:
battle = pd.read_csv('battle.csv')

battle.to_sql(
    name='battle',
    con=sqlite_conn,
    if_exists='replace',
    index=False
)

500

#### 3) 테이블 데이터 조회

- [read_sql_table()](https://pandas.pydata.org/docs/reference/api/pandas.read_sql_table.html) 메소드 → 저장된 테이블 데이터 읽어오기

In [5]:
pd.read_sql_table(
    table_name='pokemon',
    con=sqlite_conn
)

products_from_db = pd.read_sql(
    sql='SELECT * FROM pokemon',
    con=sqlite_conn
)

- [read_sql()](https://pandas.pydata.org/docs/reference/api/pandas.read_sql.html) 메소드 → SQL 쿼리 기반 데이터 조회

In [6]:
country_cnt = pd.read_sql(
    sql='SELECT type1, COUNT(*) AS cnt FROM pokemon GROUP BY type1 ORDER BY cnt DESC',
    con=sqlite_conn
)

country_cnt

,type1,cnt
0,Water,46
1,Normal,37
2,Bug,22
3,Grass,21
4,Fire,20
5,Psychic,15
6,Poison,15
7,Electric,15
8,Rock,13
9,Ground,11


#### 4) 타입을 지정하여 테이블 저장

In [ ]:
from sqlalchemy import (
    INTEGER,
    VARCHAR
)

In [ ]:
trainer.to_sql(
    name='trainer',
    con=sqlite_conn,
    if_exists='replace',
    index=False,
    dtype={
        'id': INTEGER,
        'name': VARCHAR(50),
        'age': INTEGER,
        'hometown': VARCHAR(50),
        'prefer_type': VARCHAR(20),
        'badge_count': INTEGER,
        'achievement_level': VARCHAR(20)
        }
    )